# 04 — Hypothesis Testing

**Research Question yang dibahas:**
- RQ2: Apakah rata-rata jumlah issue per minggu berubah secara signifikan setelah rilis pandas 2.0?

**Member:** [Nama Kamu] — Hypothesis Analyst
**Tujuan notebook ini:** Menguji secara formal (menggunakan Z-test) apakah perbedaan rata-rata issue per minggu sebelum dan sesudah pandas 2.0 — yang sudah diestimasi oleh Member B (notebook 02) dan diberi confidence interval oleh Member C (notebook 03) — signifikan secara statistik.

## AI Usage Disclosure

**Member:** [Nama Kamu] — Hypothesis Analyst | **Tools used:** Claude

| Task | Tool | Prompt summary | Output modified? |
| ---- | ---- | -------------- | ----------------- |
| Menulis fungsi `z_test_one_sample` dan `z_test_two_sample` di `src/hypothesis.py` | Claude | "Implementasikan one-sample dan two-sample Z-test sesuai rumus Tsun (2020) p.306 dan p.309, dengan output dict berisi z_stat, p_value, decision, interpretation" | [isi: Ya/Tidak — jelaskan jika ada perubahan] |
| Menyusun kerangka/struktur notebook ini (urutan sel, langkah 1-6) | Claude | "Bantu susun struktur notebook 04 dengan prosedur 6 langkah uji hipotesis" | [isi: Ya/Tidak] |

**Written entirely without AI:** Rumusan H0 dan Ha pada Langkah 1 (Bagian 1 dan Bagian 2), semua cell interpretasi/kesimpulan kontekstual, dan Ringkasan & Handoff.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from hypothesis import z_test_one_sample, z_test_two_sample

plt.rcParams['figure.figsize']    = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

# Load hasil estimasi dari Member B (handoff dari notebook 02)
est = pd.read_csv('../data/clean/estimation_summary.csv', index_col=0)['nilai']

lambda_pre  = est['lambda_hat_pre_v2']
lambda_post = est['lambda_hat_post_v2']
n_pre  = int(est['n_weeks_pre'])
n_post = int(est['n_weeks_post'])

print('=== Input dari Member B (notebook 02) & Member C (notebook 03) ===')
print(f'  lambda_hat_pre_v2  = {lambda_pre:.4f}  (n = {n_pre} minggu)')
print(f'  lambda_hat_post_v2 = {lambda_post:.4f}  (n = {n_post} minggu)')
print('  (Member C sudah menunjukkan bahwa CI 95% kedua periode TIDAK overlap)')

---
# BAGIAN 1 — ONE-SAMPLE Z-TEST
### Apakah rata-rata issue/minggu sesudah pandas 2.0 berbeda dari rata-rata historis (sebelum pandas 2.0)?

Di sini, rata-rata sebelum pandas 2.0 (`lambda_hat_pre_v2`) digunakan sebagai nilai acuan **μ₀** (nilai historis / status quo), dan kita uji apakah rata-rata sesudah pandas 2.0 (`lambda_hat_post_v2`) berbeda secara signifikan dari μ₀.

## Langkah 1 — Rumuskan H0 dan Ha

[TUGAS KAMU — tulis H0 dan Ha di sini dengan kalimatmu sendiri dan dalam notasi matematis]

Contoh kerangka (lengkapi dan sesuaikan sendiri, JANGAN disalin mentah):
- H0: μ = ... (nilai apa yang menjadi acuan/status quo?)
- Ha: μ ... μ₀ (apakah dugaanmu arahnya "berkurang", "berbeda", atau "meningkat"? ini menentukan apakah testnya one-sided atau two-sided)

> Catatan: pilihan arah Ha akan menentukan nilai `alternative` ('less', 'greater', atau 'two-sided') pada kode di Langkah 3-5. Pastikan keduanya konsisten.

## Langkah 2 — Tentukan Tingkat Signifikansi (α)

Kita gunakan α = 0.05 (standar dalam ilmu statistik), yaitu kita bersedia menerima risiko 5% melakukan Type I Error (menolak H0 padahal H0 benar).

In [ ]:
alpha = 0.05
print(f'Tingkat signifikansi (alpha) = {alpha}')

## Langkah 3-5 — Hitung Statistik Uji, p-value, dan Keputusan

**Formula one-sample Z-test** (Tsun, 2020, p. 306):
$$z = \frac{\bar{x} - \mu_0}{\sigma / \sqrt{n}}$$

di mana:
- $\bar{x}$ = `lambda_hat_post_v2` (rata-rata sampel periode sesudah pandas 2.0)
- $\mu_0$ = `lambda_hat_pre_v2` (nilai acuan/historis)
- $\sigma$ = $\sqrt{\bar{x}}$ (untuk distribusi Poisson, Var(X) = E[X] = λ, jadi simpangan baku diestimasi dari sampel itu sendiri)
- $n$ = jumlah minggu pada periode sesudah pandas 2.0

**PENTING:** Sesuaikan nilai `alternative` di bawah ini ('less', 'greater', atau 'two-sided') agar konsisten dengan Ha yang kamu tulis di Langkah 1.

In [ ]:
x_bar = lambda_post
mu0   = lambda_pre
sigma = np.sqrt(lambda_post)   # sigma diestimasi dari data sampel (post pandas 2.0)
n     = n_post

alternative = 'less'  # TODO: ganti sesuai arah Ha kamu di Langkah 1 ('less' / 'greater' / 'two-sided')

hasil_one_sample = z_test_one_sample(x_bar=x_bar, mu0=mu0, sigma=sigma, n=n,
                                      alternative=alternative, alpha=alpha)

print('=== One-Sample Z-Test ===')
print(f"  x_bar (lambda_post) : {x_bar:.4f}")
print(f"  mu0 (lambda_pre)    : {mu0:.4f}")
print(f"  sigma               : {sigma:.4f}")
print(f"  n                   : {n}")
print(f"  alternative         : {alternative}")
print(f"  z_stat              : {hasil_one_sample['z_stat']:.4f}")
print(f"  p_value             : {hasil_one_sample['p_value']:.6e}")
print(f"  decision            : {hasil_one_sample['decision']}")

## Langkah 6 — Interpretasi & Kesimpulan

[TUGAS KAMU — tulis interpretasi di sini]

Panduan (hapus setelah ditulis ulang dengan kalimatmu sendiri):
- Sebutkan keputusan: **"reject H0"** atau **"fail to reject H0"** — JANGAN PERNAH menulis "accept H0".
- Jelaskan apa arti keputusan ini dalam konteks proyek pandas (apakah rata-rata issue/minggu sesudah pandas 2.0 berbeda signifikan dari rata-rata historis, dan ke arah mana?).
- Hubungkan dengan hasil Member C di notebook 03 — apakah hasil uji ini konsisten dengan kedua confidence interval yang tidak overlap?
- Sebutkan keterbatasan: misalnya asumsi normal-approximation untuk Poisson dengan n yang relatif kecil (n=37 untuk periode sesudah pandas 2.0).

---
# BAGIAN 2 — TWO-SAMPLE Z-TEST
### Apakah rata-rata issue/minggu sebelum dan sesudah pandas 2.0 berbeda secara signifikan?

Berbeda dengan Bagian 1 (yang membandingkan satu periode terhadap nilai acuan tetap), di sini kita membandingkan **dua sampel independen** secara langsung: rata-rata issue/minggu periode sebelum pandas 2.0 vs periode sesudah.

## Langkah 1 — Rumuskan H0 dan Ha

[TUGAS KAMU — tulis H0 dan Ha di sini dengan kalimatmu sendiri dan dalam notasi matematis]

Contoh kerangka (lengkapi dan sesuaikan sendiri, JANGAN disalin mentah):
- H0: μ_pre ... μ_post (hubungan apa yang diasumsikan di bawah H0?)
- Ha: μ_pre ... μ_post (apakah dugaanmu μ_pre lebih besar, lebih kecil, atau sekadar berbeda dari μ_post?)

> Catatan: pilihan arah Ha akan menentukan nilai `alternative` pada kode di Langkah 3-5. Pastikan keduanya konsisten, dan idealnya konsisten juga secara logis dengan H0/Ha di Bagian 1.

## Langkah 2 — Tentukan Tingkat Signifikansi (α)

Kita tetap menggunakan α = 0.05, konsisten dengan Bagian 1.

In [ ]:
alpha = 0.05
print(f'Tingkat signifikansi (alpha) = {alpha}')

## Langkah 3-5 — Hitung Statistik Uji, p-value, dan Keputusan

**Formula two-sample Z-test** (Tsun, 2020, p. 309):
$$z = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\dfrac{\sigma_1^2}{n_1} + \dfrac{\sigma_2^2}{n_2}}}$$

di mana:
- $\bar{x}_1$ = `lambda_hat_pre_v2`, $\sigma_1 = \sqrt{\bar{x}_1}$, $n_1$ = jumlah minggu sebelum pandas 2.0
- $\bar{x}_2$ = `lambda_hat_post_v2`, $\sigma_2 = \sqrt{\bar{x}_2}$, $n_2$ = jumlah minggu sesudah pandas 2.0

**PENTING:** Sesuaikan nilai `alternative` di bawah ini agar konsisten dengan Ha yang kamu tulis di Langkah 1 (perhatikan urutan x_bar1 dan x_bar2 — `alternative='greater'` menguji Ha: μ1 > μ2).

In [ ]:
x_bar1, x_bar2 = lambda_pre, lambda_post
sigma1, sigma2 = np.sqrt(lambda_pre), np.sqrt(lambda_post)
n1, n2 = n_pre, n_post

alternative = 'greater'  # TODO: ganti sesuai arah Ha kamu di Langkah 1 ('less' / 'greater' / 'two-sided')

hasil_two_sample = z_test_two_sample(x_bar1=x_bar1, x_bar2=x_bar2,
                                      sigma1=sigma1, sigma2=sigma2,
                                      n1=n1, n2=n2,
                                      alternative=alternative, alpha=alpha)

print('=== Two-Sample Z-Test ===')
print(f"  x_bar1 (lambda_pre)  : {x_bar1:.4f}  (n1 = {n1})")
print(f"  x_bar2 (lambda_post) : {x_bar2:.4f}  (n2 = {n2})")
print(f"  alternative          : {alternative}")
print(f"  z_stat               : {hasil_two_sample['z_stat']:.4f}")
print(f"  p_value              : {hasil_two_sample['p_value']:.6e}")
print(f"  decision             : {hasil_two_sample['decision']}")

In [ ]:
# Visualisasi: distribusi null (N(0,1)) dan posisi z_stat
z_range = np.linspace(-25, 25, 1000)
from scipy.stats import norm
pdf_null = norm.pdf(z_range)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(z_range, pdf_null, color='steelblue', label='Distribusi H0 (Z ~ N(0,1))')
ax.axvline(hasil_two_sample['z_stat'], color='red', linestyle='--',
           label=f"z_stat = {hasil_two_sample['z_stat']:.2f}")
ax.axvline(norm.ppf(1 - alpha), color='green', linestyle=':',
           label=f"Batas kritis (alpha={alpha}, one-sided)")
ax.set_title('Two-Sample Z-Test — Posisi Statistik Uji terhadap Distribusi Null')
ax.set_xlabel('z')
ax.set_ylabel('Densitas')
ax.legend()
plt.tight_layout()
plt.savefig('../data/clean/plot_ztest_two_sample.png', dpi=150)
plt.show()

## Langkah 6 — Interpretasi & Kesimpulan

[TUGAS KAMU — tulis interpretasi di sini]

Panduan (hapus setelah ditulis ulang dengan kalimatmu sendiri):
- Sebutkan keputusan: **"reject H0"** atau **"fail to reject H0"** — JANGAN PERNAH menulis "accept H0".
- Jelaskan makna keputusan ini secara konkret untuk proyek pandas: apakah penurunan rata-rata issue/minggu dari ~22 menjadi ~10,6 setelah pandas 2.0 adalah perbedaan yang nyata secara statistik, atau bisa jadi hanya kebetulan/variasi sampel?
- Hubungkan dengan hasil Bagian 1 — apakah kedua uji memberi kesimpulan yang konsisten?
- Diskusikan kemungkinan penjelasan di balik hasil ini (misalnya: rilis besar pandas 2.0 menyelesaikan banyak isu lama, perubahan kebijakan triase issue, dll — sesuai temuan EDA Member A jika relevan).

---
## Ringkasan & Handoff ke Layer Berikutnya

**Ringkasan temuan uji hipotesis:** [TUGAS KAMU — tulis 2-3 poin ringkasan dari Bagian 1 dan Bagian 2]

**Output untuk layer berikutnya (Member E — Computational Analysis):**
- Hasil uji hipotesis di notebook ini menunjukkan bahwa perbedaan rata-rata issue/minggu sebelum vs sesudah pandas 2.0 [signifikan/tidak signifikan — isi sesuai hasil di atas].
- Member E dapat menggunakan temuan ini sebagai konteks untuk simulasi Monte Carlo (misalnya terkait RQ3: probabilitas sebuah issue membutuhkan lebih dari 30 hari untuk ditutup), dan/atau untuk MCMC/Bloom Filter sesuai rancangan layer Week 14.